# Atención a mano: Q, K, V y la máscara causal

**Facsímil 3 · Arquitecturas y modelos** — capítulos 2 y 3
(el Transformer por dentro; *queries*, *keys*, *values*, máscara causal y *softmax*).

«Atención» es la palabra más repetida y peor entendida de la IA moderna. En este cuaderno deja de ser
un eslogan: la calculas **a mano**, con NumPy, hasta el último número. Vas a ver cómo cada palabra
«mira» a las demás, cómo se reparte ese foco con una *softmax*, y por qué la **máscara causal** impide
que una palabra haga trampas mirando el futuro. Sin librerías mágicas: cuatro multiplicaciones de
matrices. Entender esto es entender el 80% de por qué funciona un Transformer, y por tanto cualquier
LLM.

### Qué vas a aprender
- Qué problema resuelve la atención que las redes anteriores (RNN) resolvían mal.
- Qué son exactamente la *query*, la *key* y el *value*, con una analogía de **búsqueda**.
- Cómo un **producto escalar** mide parecido, calculado palabra por palabra a mano.
- Por qué se divide por $\sqrt{d}$, **midiéndolo** con un experimento (no de palabra).
- Qué hace la **máscara causal** y por qué separa un modelo que *genera* de uno que *entiende*.
- Un **ejemplo interpretable** donde la atención resuelve una ambigüedad de verdad.
- Qué es la **atención multi-cabeza** y por qué varias miradas ven más que una.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves. Lee cada celda de texto antes de correr la de código.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(1)

frase = ["El", "gato", "que", "duerme", "ronronea"]
n, d = len(frase), 16          # 5 palabras, 16 dimensiones por vector
# En un modelo real, Q, K, V salen de multiplicar el embedding por matrices APRENDIDAS.
# Aqui los simulamos con matrices aleatorias para ver la mecanica con claridad.
X = np.random.randn(n, d)
Wq, Wk, Wv = (np.random.randn(d, d) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
print("Cada palabra es un vector. Formas de Q, K, V:", Q.shape)
print("Asi que toda la frase cabe en una matriz de", n, "filas x", d, "columnas.")


## 1. El problema que resuelve la atención

Antes de los Transformers, para procesar una frase se usaban redes recurrentes (RNN): leían palabra a
palabra, arrastrando un «resumen» de lo anterior. Tenían dos problemas: eran **lentas** (no se pueden
paralelizar, hay que ir en orden) y se les **olvidaba** lo lejano (la primera palabra apenas influía en
la última, las frases largas se difuminaban).

La atención da la vuelta a la idea: en vez de arrastrar un resumen, deja que **cada palabra mire
directamente a todas las demás a la vez** y decida de cuáles fiarse. Es paralelizable (todas las
miradas se calculan de golpe) y no sufre de distancia (la palabra 1 y la 100 se conectan igual de
fácil). Ese cambio es lo que hizo posibles los LLM. Vamos a construir ese «mirar».


### Una analogía cotidiana: la mesa de reunión

Imagina una mesa con cinco personas (las cinco palabras). Cada una, antes de hablar, lanza una
**pregunta** al aire («¿quién sabe de gatos?»), y todas las demás llevan colgado un **cartel** que
resume de qué saben. Cada persona mira los carteles, decide a quién hacer caso (más a quien encaja con
su pregunta, menos a quien no) y se queda con una **mezcla** de lo que dicen los elegidos. La pregunta
es la *query*, el cartel es la *key* y lo que cada uno aporta cuando le eligen es el *value*. La
atención es exactamente eso, pero con vectores y multiplicaciones.


## 2. Q, K, V: una búsqueda, palabra por palabra

La mejor analogía técnica es una **búsqueda**. Cada palabra genera tres vectores a partir de su
embedding:

- una **query** ($Q$): «esto es lo que ando buscando»;
- una **key** ($K$): «esto es lo que yo ofrezco»;
- un **value** ($V$): «esta es la información que aporto si me eliges».

El parecido entre la *query* de una palabra y la *key* de otra (su producto escalar) mide **cuánto le
interesa** mirarla. Es como teclear una búsqueda (query) y compararla con los títulos de los documentos
(keys) para decidir cuáles leer (values). Calculamos esa tabla de parecidos.


In [ ]:
scores = Q @ K.T / np.sqrt(d)
print("Tabla de afinidades (fila = quien pregunta, columna = a quien mira):\n")
print("            " + "  ".join(f"{w:>8}" for w in frase))
for i, w in enumerate(frase):
    print(f"{w:>10}  " + "  ".join(f"{scores[i,j]:>8.2f}" for j in range(n)))


## 3. Un producto escalar a mano: qué significa «parecido»

Esa tabla son productos escalares. El producto escalar de dos vectores es grande cuando **apuntan en
la misma dirección** y pequeño (o negativo) cuando apuntan a lados distintos. No hay nada más: es
sumar, componente a componente, el producto de sus coordenadas. Lo hacemos a mano para un par de
palabras y comprobamos que coincide con lo que dio NumPy.


In [ ]:
i, j = 1, 0     # la query de 'gato' contra la key de 'El'
a_mano = sum(Q[i, t] * K[j, t] for t in range(d))   # producto escalar definicion pura
de_numpy = Q[i] @ K[j]
print(f"Producto escalar Q['{frase[i]}'] . K['{frase[j]}']:")
print(f"  sumando a mano los {d} productos: {a_mano:8.4f}")
print(f"  con el operador @ de NumPy     : {de_numpy:8.4f}")
print(f"  (coinciden: {np.isclose(a_mano, de_numpy)})\n")

# Comparamos dos parejas para ver que 'mas alineado' = score mas alto
pares = [(4, 1), (4, 3)]   # 'ronronea' mirando a 'gato' vs a 'duerme'
for (a, b) in pares:
    print(f"  '{frase[a]}' mira a '{frase[b]}': afinidad sin escalar = {Q[a] @ K[b]:7.3f}")
print("\nUn numero mas alto = esas dos palabras 'encajan' mas en este espacio aprendido.")


## 4. ¿Por qué dividir por $\sqrt{d}$? Vamos a medirlo

Fíjate en el `/ np.sqrt(d)` de la celda de afinidades. No es un capricho, y no hace falta creérselo: lo
**medimos**. El producto escalar de dos vectores de dimensión $d$ crece, en magnitud, con $d$ (sumamos
más términos). Si no lo corrigiéramos, con vectores grandes los *scores* serían enormes, la *softmax*
(que viene después) se saturaría y daría casi todo el peso a una sola palabra, con gradientes
minúsculos que impedirían aprender. Comparamos la dispersión de los *scores* para $d=16$ y $d=64$, con
y sin escalado.


In [ ]:
def scores_crudos(d_dim, escalar, seed=0):
    rng = np.random.default_rng(seed)
    # Tomamos Q y K como vectores 'tipicos' (cada componente con varianza 1):
    Ql, Kl = rng.standard_normal((n, d_dim)), rng.standard_normal((n, d_dim))
    s = Ql @ Kl.T
    return s / np.sqrt(d_dim) if escalar else s

print(f"{'dimension d':>12} | {'sin /sqrt(d)':>14} | {'con /sqrt(d)':>14}   (desviacion tipica de los scores)")
for d_dim in [16, 64, 256]:
    sin_e = scores_crudos(d_dim, False).std()
    con_e = scores_crudos(d_dim, True).std()
    print(f"{d_dim:>12} | {sin_e:>14.2f} | {con_e:>14.2f}")
print("\nSin escalar, la dispersion se dispara al subir d. Con /sqrt(d) se queda en un rango estable.")
print("Esa estabilidad es justo lo que evita que la softmax se sature.")


## 5. La máscara causal: no mirar el futuro

Cuando un modelo **genera** texto, lo hace palabra a palabra de izquierda a derecha. Al decidir la
palabra 2, no puede mirar la 3, 4, 5: aún no existen. Si las mirara durante el entrenamiento, haría
trampa (vería la respuesta). Lo impedimos poniendo $-\infty$ en la mitad superior de la tabla (el
futuro). Así, al normalizar, esas casillas se vuelven cero: imposible mirar hacia delante.


In [ ]:
mascara = np.triu(np.ones((n, n)), k=1).astype(bool)   # triangulo superior = futuro
scores_causal = scores.copy()
scores_causal[mascara] = -np.inf
print("Con el futuro tapado (-inf = prohibido mirar):\n")
print("            " + "  ".join(f"{w:>8}" for w in frase))
for i, w in enumerate(frase):
    fila = "  ".join(("    -inf" if scores_causal[i,j]==-np.inf else f"{scores_causal[i,j]:>8.2f}") for j in range(n))
    print(f"{w:>10}  " + fila)


## 6. Softmax: repartir el 100% de la atención

La *softmax* convierte cada fila en porcentajes que suman 1: cuánta atención dedica cada palabra a cada
una de las anteriores (y a sí misma). Para cada fila $i$, $\text{softmax}(s)_j = e^{s_j} / \sum_k
e^{s_k}$. Esto ya es **el mapa de atención**. Lo imprimimos, comprobamos que cada fila suma 1 y lo
dibujamos.


In [ ]:
def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)   # estabilidad numerica
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

pesos = softmax(scores_causal)
print("Reparto de atencion por fila (cada fila suma 1):\n")
print("            " + "  ".join(f"{w:>8}" for w in frase))
for i, w in enumerate(frase):
    print(f"{w:>10}  " + "  ".join(f"{pesos[i,j]:>8.2f}" for j in range(n)))
print("\nComprobacion: suma de cada fila =", np.round(pesos.sum(axis=1), 3),
      "-> todas valen 1, como debe ser.")

plt.figure(figsize=(4.6, 4))
plt.imshow(pesos, cmap="Greys", vmin=0, vmax=1)
plt.xticks(range(n), frase, rotation=45, ha="right"); plt.yticks(range(n), frase)
plt.xlabel("mira a"); plt.ylabel("palabra que pregunta")
plt.title("Mapa de atencion (con mascara causal)")
plt.colorbar(fraction=0.046); plt.tight_layout(); plt.show()


**Lee el mapa.** Es triangular: la mitad de arriba a la derecha está en blanco (cero), porque ninguna
palabra mira a las que vienen después. Cada fila reparte su atención solo entre ella y las anteriores.
Eso es exactamente lo que hace un LLM al generar palabra a palabra.


## 7. Sin máscara: la atención bidireccional (GPT frente a BERT)

¿Y si quitamos la máscara? Entonces cada palabra puede mirar a **todas**, también a las posteriores.
Eso es la atención *bidireccional* de modelos como BERT: muy buena para *entender* (clasificar,
extraer, rellenar huecos), porque al decidir sobre una palabra aprovecha todo el contexto, antes y
después. Pero inservible para *generar* de izquierda a derecha, porque haría trampa. Misma maquinaria,
una sola diferencia: la máscara. Dibujamos los dos mapas lado a lado.


In [ ]:
pesos_bi = softmax(scores)            # sin tapar el futuro
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (titulo, M) in zip(axes, [("causal (GPT): solo el pasado", pesos),
                                   ("bidireccional (BERT): todo", pesos_bi)]):
    ax.imshow(M, cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(frase, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(frase, fontsize=7)
    ax.set_title(titulo, fontsize=9)
plt.tight_layout(); plt.show()

llenas_causal = int((pesos > 1e-6).sum())
llenas_bi = int((pesos_bi > 1e-6).sum())
print(f"Casillas con atencion > 0: causal = {llenas_causal}, bidireccional = {llenas_bi}.")
print("La bidireccional llena toda la cuadricula; la causal solo el triangulo del pasado.")


## 8. Mezclar los valores: la salida de la capa

Por último, cada palabra se queda con una **mezcla de los valores** ($V$) de las palabras que mira,
pesada por su atención: $\text{salida} = \text{pesos} \cdot V$. Ese vector mezclado es lo que sale de la
capa de atención y sigue hacia adelante en el Transformer. La primera palabra solo puede atenderse a sí
misma (la máscara le tapa todo lo demás); las posteriores reparten entre lo que sí pueden ver.


In [ ]:
salida = pesos @ V
print("Salida de la capa de atencion:", salida.shape, "\n")
print("Reparto de atencion de cada palabra (los ceros a la derecha son el futuro tapado):")
for i, w in enumerate(frase):
    print(f"  {w:>9}: {np.round(pesos[i], 2)}")
print(f"\nLa primera palabra ('{frase[0]}') solo puede atenderse a si misma (peso {pesos[0,0]:.2f}): la")
print("mascara le tapa todo lo demas. Las siguientes reparten entre lo que SI pueden ver, segun parecido.")


### La salida es, literalmente, un promedio ponderado

Que `salida = pesos @ V` no es una fórmula que haya que creerse: la salida de cada palabra es la **media
de los *values***, donde cada *value* pesa lo que diga su atención. Lo comprobamos a mano para una
palabra: sumamos los *values* multiplicados por sus pesos y vemos que da exactamente la fila de la
salida.


In [ ]:
i = 3   # 'duerme'
a_mano = sum(pesos[i, j] * V[j] for j in range(n))   # promedio ponderado de los values
print(f"Salida de '{frase[i]}' calculada de dos maneras (primeras 5 dimensiones):")
print("  pesos[i] @ V        :", np.round(salida[i, :5], 4))
print("  suma ponderada a mano:", np.round(a_mano[:5], 4))
print(f"  coinciden por completo: {np.allclose(a_mano, salida[i])}")
print(f"\n'{frase[i]}' reparte su atencion asi: {np.round(pesos[i], 2)},")
print("y su salida es esa mezcla de los values de las palabras que mira. Nada mas.")


### Una nota sobre el tamaño real

Aquí la frase tiene 5 palabras, así que la tabla de atención es de 5x5. En un LLM real, esa tabla es de
**(longitud del texto) x (longitud del texto)**: con 2.000 *tokens* de contexto son 4 millones de
casillas por cabeza y por capa. De ahí que la **ventana de contexto** sea cara (crece al cuadrado) y que
buena parte de la investigación reciente busque atenciones más baratas. La mecánica que has calculado a
mano es la misma; solo cambia la escala.


## 9. Un ejemplo interpretable: la atención resuelve una ambigüedad

Con vectores aleatorios la mecánica se ve, pero los parecidos no «significan» nada. Montemos un caso
**diseñado a mano** para que sí signifique. La palabra «banco» es ambigua: ¿el de sentarse o el del
dinero? Su sentido depende del contexto. Damos a cada palabra una *key* que apunta a una dirección
distinta (un «tema») y hacemos que la *query* de «banco» apunte sobre todo hacia el tema «dinero». La
atención debería concentrarse ahí. No es aleatorio: el resultado es exacto y comprobable.


In [ ]:
contexto = ["río", "dinero", "sentarse", "pan"]
# Cada key es una direccion-tema distinta (vectores de la base canonica):
K2 = np.eye(4)
# La query de 'banco' busca sobre todo el tema 'dinero' (indice 1), con algo de ruido:
q_banco = np.array([0.3, 1.0, 0.2, 0.1])
afin = q_banco @ K2.T                       # parecido con cada tema
w2 = np.exp(afin) / np.exp(afin).sum()      # softmax (1 sola fila)
print("Hacia que palabra del contexto mira 'banco':")
for w, p in sorted(zip(contexto, w2), key=lambda t: -t[1]):
    print(f"  {w:>10}: {p*100:5.1f}%  {'#'*int(p*40)}")
ganadora = contexto[int(np.argmax(w2))]
print(f"\nGana '{ganadora}': la atencion ha desambiguado 'banco' hacia el sentido financiero.")
print("Cambia el vector q_banco hacia el indice 2 y 'banco' pasara a mirar a 'sentarse'.")


Esto es, en miniatura, lo que ocurre dentro de un LLM real: las *keys* y las *queries* las **aprende**
el modelo, y acaban codificando relaciones útiles (género, tema, sintaxis). La atención es el mecanismo
por el que «que duerme» se conecta con «gato» y no con «ronronea». Aquí lo hemos puesto a mano para
verlo; en el modelo emerge del entrenamiento.


## 10. Varias miradas: atención multi-cabeza

Una sola atención reparte un único foco. Pero una palabra puede querer mirar a otras por **motivos
distintos** a la vez (concordancia de género, sujeto del verbo, referencia de un pronombre...). Por eso
los Transformers usan **varias cabezas** en paralelo, cada una con sus propias matrices $W_q, W_k, W_v$,
y luego **combinan** sus salidas (las concatenan y las pasan por una proyección $W_o$). Simulamos
varias cabezas, vemos que cada una produce un mapa **diferente** y juntamos sus salidas.


In [ ]:
def una_cabeza(X, d):
    Wq, Wk, Wv = (np.random.randn(d, d) for _ in range(3))
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    s = Q @ K.T / np.sqrt(d)
    s[np.triu(np.ones((n,n)), k=1).astype(bool)] = -np.inf
    return softmax(s)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
mapas = []
for h, ax in enumerate(axes):
    pesos_h = una_cabeza(X, d)
    mapas.append(pesos_h)
    ax.imshow(pesos_h, cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(frase, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(frase, fontsize=7)
    ax.set_title(f"cabeza {h+1}")
plt.tight_layout(); plt.show()
print("Tres cabezas, tres mapas distintos: cada una aprende a mirar un tipo de relacion.")


### ¿De verdad miran cosas distintas, o se parecen?

«Distintos» a ojo es fácil de decir. Pongámosle número: medimos cuánto se parecen los mapas de las
cabezas comparando, casilla a casilla (solo donde la máscara permite mirar), su correlación media. Y
combinamos las salidas como hace un Transformer real: concatenar las cabezas y proyectar.


In [ ]:
tril = ~np.triu(np.ones((n, n)), k=1).astype(bool)   # casillas permitidas (pasado + diagonal)
vecs = [m[tril] for m in mapas]
cors = [np.corrcoef(vecs[a], vecs[b])[0, 1] for a in range(3) for b in range(a+1, 3)]
print(f"Correlacion media entre los mapas de las cabezas: {np.mean(cors):.2f}")
print("(1.0 seria que son identicas; valores bajos = cada cabeza mira algo distinto).\n")

# Combinar varias cabezas: concatenar sus salidas y proyectar con Wo
d_cab, H = 8, 4
def salida_cabeza(X):
    Wq, Wk, Wv = (np.random.randn(d, d_cab) for _ in range(3))
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    s = Q @ K.T / np.sqrt(d_cab)
    s[np.triu(np.ones((n, n)), k=1).astype(bool)] = -np.inf
    return softmax(s) @ V                      # (n, d_cab)

concat = np.concatenate([salida_cabeza(X) for _ in range(H)], axis=1)   # (n, H*d_cab)
Wo = np.random.randn(H * d_cab, d)
multi = concat @ Wo                            # (n, d): vuelve al tamano original
print(f"{H} cabezas de {d_cab} dims -> concatenadas: {concat.shape} -> tras proyectar Wo: {multi.shape}")
print("La salida vuelve a tener el tamano de entrada, lista para la siguiente capa del Transformer.")


## 11. Pruébalo tú

1. **Quita la máscara** en la celda de la *softmax* (usa `scores` en vez de `scores_causal`). El mapa se
   llena: atención bidireccional. ¿Por qué eso sirve para clasificar pero no para generar?
2. **Sube la dimensión** `d` a 64 y vuelve a ejecutar: los repartos cambian, la mecánica no.
3. **Rompe el escalado:** en la celda de afinidades quita el `/ np.sqrt(d)` y mira los pesos. ¿Se
   concentran casi todos en una palabra? Acabas de ver por qué el escalado existe.
4. **Desambigua tú:** en el ejemplo de «banco», cambia `q_banco` para que apunte al índice 0 («río») o
   al 2 («sentarse»). ¿La atención sigue a tu query?
5. **Compara las cabezas:** ¿la correlación media entre mapas es alta o baja? Esa diversidad es la razón
   de usar varias. Sube `H` a 8 cabezas y mira cómo cambia la forma concatenada.
6. **Una frase tuya:** cambia la lista `frase` por otra de tu cosecha (cuidando que `n` cuadre) y observa
   el triángulo del mapa.


## 12. Errores comunes

- **Pensar que Q, K, V son cosas mágicas.** Son tres proyecciones lineales del embedding; lo que las
  hace útiles es que sus matrices se *aprenden*.
- **Creer que el producto escalar mide «significado».** Mide **alineación de direcciones** en un espacio
  aprendido. Que esa alineación capture significado es mérito del entrenamiento, no del operador.
- **Olvidar el escalado $\sqrt{d}$.** Sin él, la dispersión de los *scores* crece con $d$, la *softmax*
  se satura y el modelo no aprende bien. Lo has medido, no es teoría.
- **Confundir atención causal y bidireccional.** La máscara es lo único que separa un GPT (genera) de un
  BERT (entiende). Misma maquinaria, distinta máscara.
- **Creer que una cabeza basta.** Varias cabezas capturan relaciones distintas y luego se combinan; es
  parte de por qué los Transformers son tan expresivos.


## 13. Qué te llevas

- La atención es **un reparto de pesos**: cada palabra decide cuánto mira a cada otra, vía parecido
  *query*-*key* (un producto escalar), normalizado con *softmax*, y se lleva una mezcla de los *values*.
- El **escalado** $\sqrt{d}$ no es decorativo: mantiene los *scores* en un rango sano para que la
  *softmax* no colapse. Lo viste crecer con $d$ en una tabla real.
- La **máscara causal** es lo que distingue a un modelo que *genera* (solo mira el pasado) de uno que
  *entiende* (mira todo). Una matriz triangular, nada más.
- La **multi-cabeza** son varias miradas distintas que se concatenan y se proyectan: más expresividad
  por el mismo precio conceptual.
- Apilando muchas de estas capas se construye un Transformer (capítulo 4: MLP, residual, *logits*).

**Para seguir:** el capítulo 4 termina el bloque (de la atención a los *logits* y el *sampling*); y el
siguiente cuaderno de este facsímil usa esos *logits* para generar texto.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*